In [ ]:
/*
========================================================
HANDLING FAILURES IN DISTRIBUTED SYSTEMS - SIMPLE HINGLISH NOTES
========================================================

INTRO - FAILURES NORMAL HAIN
--------------------------------------------------------
Distributed systems mein failures HONA hi hai, ye rare 
event nahi hai. Hardware fail hota, network partition hota, 
software mein bugs hote, dependencies down ho jaati hain.

EXAMPLE: Ek slow payment service (peak traffic mein 
timeout ho jaye), orders checkout mein stuck ho sakte, aur 
chhota problem bada outage ban sakta.

MAIN BAAT: Resilient system wo hai jo failure ko CONTAIN 
karta - blast radius chhota rakhta, correctness maintain 
karta, aur important user flows ko protect karta.


WHY FAILURES INEVITABLE HAIN
--------------------------------------------------------
Distributed system mein bahut saare components network pe 
communicate karte hain. Har component alag se fail ho 
sakta, aur network khud bhi unreliable hai.

TEXT DIAGRAM - E-COMMERCE CHECKOUT FLOW:

  Client -> API Gateway -> Order Service -> Payment Service -> Bank API
                                |
                          Inventory Service
                                |
                          Notification Service

ISME MULTIPLE FAILURE POINTS HAIN:
- Network failures    -> Packets lost, connections timeout
- Hardware failures   -> Servers crash, disks fail
- Software failures   -> Bugs, memory leaks, deadlocks
- Dependency failures -> External APIs down ho jaate
- Overload failures   -> Zyada traffic services ko overwhelm 
                          kar deta
- Human failures      -> Galat config, bad deployment

IMPORTANT MATH: Jitna zyada components request touch karti, 
utna kam end-to-end success rate hota. Agar har service 
99.9% available hai aur failures independent hain, to 5 
services touch karne wali request sirf ~99.5% (0.999^5) 
success hoti hai best case mein.


1. TYPES OF FAILURES
--------------------------------------------------------
Pehle failure ko classify karo. Pehle 3 types batate hain 
ki retry useful hai, dangerous hai, ya irrelevant hai. 4th 
type (cascading) batata hai ki ek failure kaise phailti hai.

--------------------------------------------------------
1.1 TRANSIENT FAILURES (Temporary, khud theek ho jaate)
--------------------------------------------------------
Same request retry karne pe success ho sakta.

EXAMPLES:
- Momentary congestion ki wajah se network timeout
- Service temporarily overloaded
- Database connection pool exhaust ho gaya
- Temporary DNS resolution failure

DEFAULT RESPONSE: Exponential backoff + jitter ke saath 
retry karo.

--------------------------------------------------------
1.2 PERMANENT FAILURES (Bina fix ke theek nahi hote)
--------------------------------------------------------
Retry karne se koi fayda nahi.

EXAMPLES:
- Invalid input data
- Resource not found (404)
- Authentication failures
- Business rule violations

DEFAULT RESPONSE: Fail fast, clear error return karo.

--------------------------------------------------------
1.3 INTERMITTENT FAILURES (Kabhi kabhi fail hote)
--------------------------------------------------------
Kabhi kaam karte, kabhi fail hote - random.

EXAMPLES:
- Flaky network connections
- Memory pressure se occasional failure
- Race conditions
- Load-dependent failures

DEFAULT RESPONSE: Limit ke saath retry karo, phir agar 
dependency baar baar fail ho to circuit breaker use karo.

--------------------------------------------------------
1.4 CASCADING FAILURES (Ek failure sab jagah phailti hai)
--------------------------------------------------------
Ye root cause nahi, propagation mode hai: ek service fail 
hoti to dependent services bhi fail ho jaati - domino 
effect.

TEXT DIAGRAM:

  Service A -> Service B -> Service C -> Service D
                                             |
                                    Everything Down!

EXAMPLES:
- Database overload sab dependent services tak phail jata
- Ek slow service caller ke thread pools khali kar deta
- Network partition multiple services isolate kar deta

DEFAULT RESPONSE: Timeouts + circuit breakers + bulkheads + 
load shedding use karo failure ko phailne se rokne ke liye.


2. CORE FAILURE HANDLING PATTERNS
--------------------------------------------------------

--------------------------------------------------------
2.1 RETRIES WITH EXPONENTIAL BACKOFF
--------------------------------------------------------
Transient failures ke liye useful, but agar sab clients 
turant retry karein to overload aur bhi bad sakta.

Exponential backoff = retries ke beech gap badhata jata hai.

TEXT DIAGRAM:

  Attempt 1 -> FAIL -> Wait 1s
  Attempt 2 -> FAIL -> Wait 2s
  Attempt 3 -> FAIL -> Wait 4s
  Attempt 4 -> ...

PARAMETERS:

  Parameter      Description                  Typical Value
  ------------------------------------------------------------
  Initial delay  Pehla retry wait time         100ms - 1s
  Multiplier     Delay badhne ka factor        2x
  Max delay      Wait time ka cap              30s - 60s
  Max retries    Total retry attempts          3 - 5
  Jitter         Randomization (thundering     0-100% of delay
                 herd rokne ke liye)

NOTE: Jitter isliye zaroori hai kyunki agar sab clients 
same time pe retry karein to recovering dependency phir se 
overwhelm ho sakti.

GOOD RETRY CANDIDATES:
- HTTP 429 (Too Many Requests)
- HTTP 503 (Service Unavailable)
- HTTP 504 (Gateway Timeout)
- Connection timeouts
- Transient network errors

RETRY MAT KARO:
- HTTP 400 (Bad Request)
- HTTP 401/403 (Auth issues)
- HTTP 404 (Not Found)
- Business logic failures

--------------------------------------------------------
2.2 CIRCUIT BREAKERS
--------------------------------------------------------
Retries chhoti failures mein help karte. Agar dependency 
lambe time tak down/overloaded rahe, to baar baar call 
karna caller ke resources waste karta aur dependency pe 
aur load daalta.

Circuit breaker failures monitor karta aur threshold cross 
hone pe "trip open" ho jata. Open hone pe, requests turant 
fail ho jaati bina call try kiye.

TEXT DIAGRAM - 3 STATES:

  [CLOSED] --failures exceed threshold--> [OPEN]
     ^                                        |
     |                                   timeout expires
     |                                        v
     |<--test requests fail--- [HALF-OPEN] --test requests succeed-->[CLOSED]

  State      Behavior                    Transition
  ------------------------------------------------------------
  Closed     Requests normally pass       Failure rate 
                                           threshold cross -> Open
  Open       Requests turant fail hote    Timeout ke baad -> 
                                           Half-Open
  Half-Open  Limited test requests jaate  Test pass -> Closed
                                           Test fail -> Open

CONFIGURATION PARAMETERS:
- Failure threshold: Kitna % failure open trigger kare 
  (e.g., 50%)
- Window size: Kitne requests evaluate karein (e.g., last 100)
- Timeout: Open state mein kitni der rukein (e.g., 60 sec)
- Test requests: Half-open mein kitne test requests bheje 
  (e.g., 3)

KAB USE KARE:
- Dependency ka failure rate sustained high ho
- Calls threads/connections/request budget consume kar rahi 
  ho
- Caller ke paas safe fallback ho ya clear degraded 
  response de sake

--------------------------------------------------------
2.3 TIMEOUTS
--------------------------------------------------------
Har external call pe timeout hona ZAROORI hai. Timeout na 
ho to slow dependency threads/connections/slots hold kar 
sakta jab tak caller khud unhealthy na ho jaye.

TEXT DIAGRAM:

  [Total Timeout: end-to-end budget]
       |
  [Connection Timeout] -> [Read Timeout]

  Type                What It Limits           Typical Value
  ------------------------------------------------------------
  Connection timeout  TCP connection banane     1-5 sec
                       ka time
  Read timeout         Response aane tak ka      5-30 sec
                        time
  Total timeout        End-to-end (retries       30-120 sec
                        included)

TIMEOUT VALUES KAISE CHOOSE KAREIN:
1. Service ka p99 latency normal conditions mein measure 
   karo
2. Occasional slowness ke liye buffer add karo (typically 
   2-3x p99)
3. Caller ke end-to-end deadline ke andar per-call budget 
   allocate karo
4. Agar retries enabled hain to retry delays/attempts bhi 
   account karo

COMMON MISTAKE: Har jagah same timeout set kar dena. 
Critical user paths ko tighter budgets chahiye background 
jobs se zyada.

HEDGED REQUESTS (bonus tarika):
Latency-sensitive reads ke liye. Thodi delay ke baad (often 
p95 latency ke aas paas), caller dusri copy request kisi 
aur replica ko bhejta, aur jo pehle response aaye wo use 
karta. Ye tarika idempotent reads ke liye theek hai, 
side-effecting writes ke liye nahi.

--------------------------------------------------------
2.4 FALLBACKS
--------------------------------------------------------
Fallback ek degraded but correct response deta jab 
dependency unavailable ho. Sirf tab useful jab fallback 
CORRECTNESS preserve kare.

TEXT DIAGRAM:

  Request -> Call Service
     |            |
  SUCCESS      FAILURE
     |            |
  Return       Fallback Strategy:
  Result        - Return Cached Data
                - Return Default Value
                - Call Backup Service
                - Return Degraded Response

  Strategy         Kab Use Karein          Example
  ------------------------------------------------------------
  Cached data      Stale data theek hai    User profile, 
                                             product catalog
  Default value    Generic value kuch nahi  Empty list, 
                    se better hai            placeholder image
  Backup service   Redundant providers hain Multiple payment 
                                              gateways
  Degraded response Partial functionality   Products without 
                    kaafi hai                 recommendations
  Queue for later  Eventual processing OK   Order confirmation 
                                              emails

WARNING: Aise fallback mat banao jo business rules silently 
violate kare. Cached recommendations dikhana theek hai. 
Bina authorization ke payment charge karna GALAT hai.

DEAD-LETTER QUEUE: Jab kaam queue karo baad ke liye, jo 
messages baar baar process fail hote unhe dead-letter queue 
mein bhejo. Isse "poison message" main queue ko block nahi 
karega ya silently lost nahi hoga - baad mein inspect + 
replay kar sakte.

--------------------------------------------------------
2.5 BULKHEADS
--------------------------------------------------------
Bulkheads resources isolate karte, taaki ek slow dependency 
dusre unrelated work ko starve na kare.

TEXT DIAGRAM:

  [Payment Service]      [Inventory Service]    [Notification Service]
         |                       |                       |
  Thread Pool: Payments   Thread Pool: Inventory  Thread Pool: Notifications
  Thread1 Thread2 Thread3 Thread1 Thread2 Thread3 Thread1 Thread2 Thread3
  (SLOW/FAILING)          (Unaffected)            (Unaffected)

  Type            Isolation Level           Use Case
  ------------------------------------------------------------
  Thread pool     Alag thread pools/service  HTTP clients, DB 
                                              connections
  Connection pool Alag connection pools      Different DBs, 
                                              external APIs
  Process         Alag processes             High vs low 
                                              priority workloads
  Container       Alag containers/pods       Microservices
  Region          Alag data centers          Geographic fault 
                                              tolerance

Bina bulkhead ke, ek slow service SAB threads khatam kar 
sakta. Bulkhead ke saath, sirf apna hi pool khatam hoga, 
baaki kaam continue karega.

--------------------------------------------------------
2.6 IDEMPOTENCY
--------------------------------------------------------
Retries tabhi safe hain jab repeat attempts duplicate side 
effects create na karein. Idempotency ensure karta ki ek hi 
logical request ko multiple baar process karna, ek baar 
process karne jaisa hi result de.

TEXT DIAGRAM - IDEMPOTENCY KEY FLOW:

  Client -> POST /charge (idempotency_key=abc123)
  Server -> Check if abc123 exists -> Not found
         -> Process payment, store abc123
  Server -> Return response
  
  [Network timeout before response reaches client]
  
  Client -> POST /charge (idempotency_key=abc123) [RETRY]
  Server -> Check if abc123 exists -> FOUND
         -> Return cached result (NO duplicate charge!)

IMPLEMENTATION PATTERNS:
- Idempotency keys: Client har logical request ke liye 
  unique ID generate karta
- Database constraints: Unique keys se duplicate inserts 
  rokna
- Conditional updates: Version numbers ya ETags use karna
- Natural idempotency: Operations ko naturally repeat-safe 
  design karna

NATURALLY IDEMPOTENT OPERATIONS:
- GET, PUT, DELETE (REST mein)
- SET key value (INCREMENT nahi)
- Upsert with unique constraint

NON-IDEMPOTENT (special handling chahiye):
- POST (naya resource create karta)
- INCREMENT counter
- Append to list
- Send email/notification

--------------------------------------------------------
2.7 GRACEFUL DEGRADATION
--------------------------------------------------------
Jab system stress mein ho, low-priority kaam chhod do 
taaki core user flows preserve rahein.

TEXT DIAGRAM:

  System Load -> Load Level?
     |
  NORMAL (< 70%)   -> Full Functionality
     - Search with filters
     - Personalized recommendations
     - Real-time inventory
     
  HIGH (~70%)      -> Reduced Features
     - Basic search
     - Generic recommendations
     - Cached inventory for browsing
     
  CRITICAL (> 90%) -> Core Features Only
     - Product listing
     - Checkout
     - Order processing

DEGRADATION LEVELS:

  Level     Trigger              Actions
  ------------------------------------------------------------
  Normal    <70% capacity        Sab features available
  High      ~70% capacity        Expensive features off 
                                  (real-time search, 
                                  personalization), cached 
                                  content serve karo
  Critical  >90% ya overloaded   Sirf core functionality 
                                  (listing, checkout, order 
                                  processing), static fallback

EXAMPLES:
- E-commerce: Browsing mein cached inventory counts serve 
  karo, recommendations disable karo, but checkout pe 
  inventory check authoritative rakho
- Social media: Comments disable karo, cached like counts 
  dikhao
- Video streaming: Video quality kam karo, autoplay 
  disable karo
- Search engine: Cached results return karo, spelling 
  suggestions band karo

IMPLEMENTATION:
- Feature flags se functionality control karo
- System metrics monitor karo (CPU, memory, queue depth)
- Threshold ke basis pe automate karo degradation decisions
- Operators ke liye manual override rakho

--------------------------------------------------------
2.8 LOAD SHEDDING AND BACKPRESSURE
--------------------------------------------------------
Graceful degradation FEATURES drop karta. Load shedding 
REQUESTS drop karta. Jab service safe capacity se aage 
nikal jaaye, aur kaam accept karna sab requests ko slow kar 
deta - poori service fail ho sakti. Kuch requests turant 
reject karna better hai, taaki baaki succeed ho sakein.

- LOAD SHEDDING: Threshold cross hone pe requests edge pe 
  hi reject/deprioritize karta, usually 429/503 turant 
  return karta, queue nahi karta
- BACKPRESSURE: Upstream callers ko signal deta "slow down" 
  - jaise queue depth bound karna, naye enqueues refuse 
  karna, concurrency limits kam karna

TEXT DIAGRAM:

  Incoming Requests -> Load above threshold?
     NO  -> Process normally
     YES -> Request priority?
            HIGH (checkout) -> Process normally
            LOW (analytics) -> Reject fast (429/503)

Shedding tabhi best kaam karta jab priority-aware ho: 
low-value traffic pehle drop karo, critical flows (checkout, 
login) protect karo.

COMMON TECHNIQUES:
- Bounded queues jo full hone pe reject karte, unlimited 
  grow nahi karte
- Concurrency limits ya token buckets per dependency
- Priority tiers - low-value traffic pehle drop
- 429/503 with Retry-After header dena (well-behaved 
  clients back off kar sakein)

NOTE: Load shedding retries + circuit breakers ke saath 
pair hota hai. Overloaded service khud ko shed karke 
protect karta, aur caller side pe backoff+jitter shed 
traffic ko turant retry storm banne se rokte.

--------------------------------------------------------
2.9 FAILOVER
--------------------------------------------------------
Failover traffic/ownership backup ko switch kar deta jab 
primary safely serve nahi kar sakta. Servers, databases, 
data centers, cloud regions - sabpe apply hota.

TEXT DIAGRAM:

  [DNS/Global LB]
      /        \
  PRIMARY      SECONDARY
  REGION       REGION
     |             |
  [Load Balancer] [Load Balancer]
  Server1,Server2  Server1,Server2
     |             |
  [Primary DB] --Replication--> [Replica DB]
       |
   FAILOVER (agar primary down ho)

FAILOVER TYPES:

  Type            Recovery Time   Data Loss Risk        Complexity
  --------------------------------------------------------------------
  Cold standby    Minutes-hours   High (periodic backup)  Low
  Warm standby    Seconds-minutes Medium (lags primary)   Medium
  Hot standby     Seconds         Low (closely synced)    High
  Active-active   Near-instant    Depends on replication  Highest
                  (agar routed                            
                  achha ho)

FAILOVER USE KARNE SE PEHLE YE SAWAL POOCHO:
1. Detection: Failure kitni jaldi detect hoti hai?
2. Decision: Kaun failover decide karta hai?
3. Execution: Switch hone mein kitna time lagta hai?
4. Data consistency: In-flight transactions ka kya hota hai?
5. Failback: Normal state mein kaise wapas aate hain?

DANGEROUS FAILURE MODE - SPLIT-BRAIN:
Jab primary AUR backup dono sochte hain ki "main hi writes 
ka owner hoon." Isko rokne ke liye:
- Quorum-based consensus
- Fencing tokens ya epochs
- Lease-based leadership

--------------------------------------------------------
2.10 DATA REPLICATION
--------------------------------------------------------
Replication data ke copies multiple nodes pe rakhta, taaki 
failure ke baad koi aur node serve kar sake.

TEXT DIAGRAM:

  Write Request -> [Leader]
                      |
              Sync    Sync    Async
             /         |          \
       [Follower1] [Follower2] [Follower3]
       
  Read Request -> Kisi bhi Replica se

REPLICATION STRATEGIES:

  Strategy       Consistency        Availability  Latency  Data Loss
  -----------------------------------------------------------------
  Synchronous    Strong (agar right Lower         Higher   Low (after 
                 replicas ack karein)                        ack)
  Asynchronous   Eventual           Higher        Lower    Possible
  Semi-sync      Async se zyada     Balanced      Medium   Possible 
                 durable                                    (before ack)

QUORUM WRITES AND READS (N replicas, W write confirmations, 
R read queries):
- W + R > N: Read/write quorums overlap karte, latest 
  acknowledged write mil sakti (agar versioning/conflict 
  resolution sahi ho)
- W = N, R = 1: Read performance maximize
- W = 1, R = N: Write performance maximize

COMMON CONFIGS (N=3 replicas maan ke):

  Configuration     N   W   R   Trade-off
  ---------------------------------------------
  Read-optimized    3   3   1   Fast reads, slow writes
  Write-optimized   3   1   3   Fast writes, slow reads
  Balanced          3   2   2   Speed aur consistency ka 
                                 balance


3. COMBINING STRATEGIES (Sab ek saath)
--------------------------------------------------------
Production mein, ye patterns LAYER mein use hote hain.

TEXT DIAGRAM - TYPICAL REQUEST PATH:

  Incoming Request
       |
  [Timeout: 5s]
       |
  [Circuit Breaker] -- Closed --> proceed
                    -- Open ----> skip call, use fallback
       |
  [Retry with Backoff] (agar transient failure ho)
       |
  [Thread Pool Bulkhead]
       |
   SUCCESS -> Return Response
   ALL FAILED -> Return Cached Data + Log Failure

TYPICAL REQUEST PATH (STEPS):
1. Timeout - kitni der wait karna hai, limit karo
2. Retry - transient failures ke liye phir try karo 
   (exponential backoff ke saath)
3. Circuit breaker - baar baar failure ke baad try karna 
   band karo
4. Bulkhead - har dependency ke liye resources isolate karo
5. Fallback - degraded functionality do
6. Monitoring - failures track karo, alert karo

IMPORTANT: ORDER MATTER KARTA HAI. Har attempt ke around 
timeout lagao, retries bounded rakho, aur circuit breaker 
final outcome observe kare taaki caller ko sustained 
failure se bacha sake.


4. IMPLEMENTATION CHECKLIST
--------------------------------------------------------

--------------------------------------------------------
4.1 DESIGN FOR FAILURE
--------------------------------------------------------
Har dependency ke liye decide karo:
- Database unavailable ho to kya hoga?
- Downstream service slow ho to kya hoga?
- Network partition ke dauran kya hoga?
- Sudden traffic spike aaye to kya hoga?

RULE: Ye INCIDENT hone se PEHLE decide karo. Outage ke 
dauran failure behavior retrofit karna slow aur risky hai.

--------------------------------------------------------
4.2 FAIL FAST
--------------------------------------------------------
Slow failures fast failures se zyada nuksan karte, kyunki 
callers wait karte waqt resources consume hote rehte.

- Short timeouts use karo
- Circuit breakers jaldi trip karo
- Input ko edge pe hi validate karo
- Overloaded ho to requests reject karo

--------------------------------------------------------
4.3 MAKE FAILURES VISIBLE
--------------------------------------------------------
Jo failure dikhta hi nahi, uska response nahi de sakte. 
Har dependency ko instrument karo:
- Har failure ko context ke saath log karo
- Per-dependency failure rates track karo
- Abnormal patterns pe alert karo
- Circuit breaker states ka dashboard banao

--------------------------------------------------------
4.4 TEST FAILURE HANDLING
--------------------------------------------------------
Failure-handling paths galat likhna easy hai kyunki wo 
stress mein chalte hain. Directly test karo:
- Fallback logic unit test karo
- Simulated failures ke saath integration test
- Production mein chaos engineering
- Regular disaster recovery drills

--------------------------------------------------------
4.5 AUTOMATE RECOVERY
--------------------------------------------------------
Common recovery paths automate karo:
- Load spikes ke liye auto-scaling
- Node failures ke liye automatic failover
- Known issues ke liye self-healing
- Complex scenarios ke liye runbooks


5. INTERVIEW ANSWER FRAMEWORK
--------------------------------------------------------
Jab interviewer poochhe "failures kaise handle karoge", 
seedha "retry" pe mat jump karo. Ye structure use karo:

1. Failure classify karo - Transient, permanent, ya 
   intermittent hai? Cascading hone ka risk hai kya?
2. Correctness pehle preserve karo - Non-idempotent side 
   effects retry mat karo, jab tak idempotency keys/
   deduplication na ho
3. Resource usage bound karo - Timeouts, retry limits, 
   circuit breakers, bulkheads use karo taaki ek dependency 
   caller ko khatam na kar de
4. Safe fallback choose karo - Cached, default, queued, ya 
   degraded response tabhi do jab business flow ke liye 
   correct ho
5. Recovery plan karo - Failover/replication ke liye 
   detection, decision authority, consistency, split-brain 
   prevention explain karo
6. Observable banao - Failure rates, timeout rates, retry 
   counts, circuit breaker state, fallback usage, recovery 
   time track karo

ACHA ANSWER KYA COVER KARTA:
- Pattern choose karne se pehle failure classify karta
- Timeouts, retry budgets, circuit breakers, bulkheads se 
  caller ko protect karta
- Pata hota kab retry NAHI karna, especially non-idempotent 
  side effects ke liye
- Safe degraded response aur business-correctness-violating 
  response mein farak karta

COMMON MISTAKES:
- Sab kuch retry karna aur accidentally overload amplify 
  karna
- Aisa fallback add karna jo correctness failure chhupa de, 
  jaise bina authorization ke payment charge kar dena
- End-to-end deadlines bhool jaana, har dependency poora 
  user request budget consume kar leti
- Failover discuss karte waqt ye na batana ki failover 
  decision kaun leta hai

SAMPLE STRONG ANSWER:
"Main pehle failure classify karunga. Transient network 
errors ke liye, bounded retries with exponential backoff + 
jitter use karunga, but har attempt end-to-end deadline ke 
andar hoga. Agar dependency unhealthy hai, circuit breaker 
aur bulkhead caller resources protect karenge. 
Side-effecting operations ke liye, retries idempotency keys 
ya deduplication maangenge. Non-critical features (jaise 
recommendations) ke liye cached/generic fallback theek hai; 
payments ya order totals ke liye, correct behavior usually 
clear failure hai, fake success nahi."


6. SUMMARY - KEY TAKEAWAYS
--------------------------------------------------------
Har failure prevent nahi ho sakti, isliye goal hai IMPACT 
CONTAIN karna: failure ko phailne se roko, data correct 
rakho, aur important workflows ko protect karo.

- Retries with backoff + jitter -> Transient failures ke 
  liye
- Timeouts -> Slow dependencies ko resources indefinitely 
  consume karne se roko
- Circuit breakers -> Sustained dependency failure mein 
  fail fast karo
- Bulkheads -> Resource pools isolate karo
- Fallbacks + graceful degradation -> Safe degraded behavior 
  ke liye
- Load shedding + backpressure -> Service ko safe capacity 
  se aage jaane pe protect karo
- Idempotency -> Side-effecting operations ke safe retries 
  ke liye
- Failover + replication -> Infrastructure aur data 
  availability ke liye

INTERVIEW TIP: Sirf pattern names mat bolo, parameters aur 
trade-offs bhi explain karo.

========================================================
END OF NOTES
========================================================
*/